# E15: Learning weights in FGW

**Started:** April 2, 2026

**Last updated:** April 2, 2026

**Research questions:** Can FGW with optimized global ortholog weights across all ortholog pairs outperform FGW with equal weighting on a predetermined set of ortholog pairs?

**Hypothesis:** Yes, optimized global ortholog weights will outperform selected constant weights because some orthologs are particularly informative while others can be counterproductive.

**Conclusion:** TBD

**Potential Next Steps:** TBD

In [4]:
from __future__ import annotations
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy as sp
from scipy.spatial.distance import cdist
from scipy import sparse
from scipy.stats import pearsonr
import ot
from ot.gromov import entropic_fused_gromov_wasserstein
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import seaborn as sns
import random
import re
from __future__ import annotations
from typing import Any, Dict, Hashable, Iterable, List, Sequence, Tuple, Union, Optional
from speciesot_helpers import label_uniform_cell_type_row, \
                              top_n_organisms_from_species, \
                              cell_types_with_n_per_organism, \
                              sample_equal_cell_types, \
                              ot_between_organisms, \
                              calc_random_mean_std, \
                              test_train_split_adata, \
                              get_cell_type_from_index, \
                              create_label_to_sample_dicts, \
                              concat_index, \
                              show_cell_type_results, \
                              cell_type_knn_acc, \
                              split_adata_by_celltype_tissue, \
                              match_cells_by_celltype_tissue, \
                              combine_transport_plans, \
                              convert_ensembl_to_gene_symbols, \
                              nearest_neighbor_cell_type, \
                              nn_gene_r2, \
                              plot_ot_transport_by_celltype, \
                              ortholog_pearson_r2_by_celltype_biomart
from speciesot_simulation_helpers import run_ot, \
                                         compute_mapping_acc, \
                                         sublabel_match_rate_from_gw, \
                                         plot_got_best_matches_with_labels, \
                                         gw_seeded, \
                                         plot_first_two_dims
import torch
from pytorch_helpers import fit_mlp
import celltypist
from celltypist import models
%matplotlib inline

### Alternating optimization code

In [2]:
def alternating_weighted_fgw_opt(
    X_mouse_pca: np.ndarray,
    X_human_pca: np.ndarray,
    orth_mouse: np.ndarray,
    orth_human: np.ndarray,
    epsilon: float,
    alpha: float,
    *,
    weight_l2: float = 1.0,
    loss_fun: str = "square_loss",
    metric: str = "euclidean",
    squared: bool = True,
    feature_loss: str = "sqdiff",
    normalize_C: bool = True,
    normalize_M: bool = True,
    p: np.ndarray | None = None,
    q: np.ndarray | None = None,
    init_w: np.ndarray | None = None,
    max_outer_iter: int = 25,
    fgw_inner_iter: int = 20,
    fgw_tol: float = 1e-9,
    outer_tol: float = 1e-6,
    solver: str = "PGD",
    sinkhorn_warmstart: bool = True,
    random_state: int | None = None,
    verbose: bool = False,
    return_log: bool = False,
):
    """
    Alternating weighted entropic FGW with quadratic-regularized weight updates.

    This alternates between:
      1) Updating the FGW coupling T with current ortholog weights w
      2) Updating w with T fixed by solving

            min_w  c^T w + (weight_l2 / 2) * ||w||_2^2
            s.t.   w >= 0, sum(w) = 1

    where c_k is the transport-weighted feature mismatch cost for ortholog k.

    The cross-space feature cost is defined as

        M_ij(w) = sum_k w_k * d(orth_mouse[i, k], orth_human[j, k]),

    which is linear in w.

    Parameters
    ----------
    X_mouse_pca : (n_mouse, d) array
        PCA coordinates for mouse cells.
    X_human_pca : (n_human, d) array
        PCA coordinates for human cells.
    orth_mouse : (n_mouse, P) array
        Ortholog expression/features for mouse.
    orth_human : (n_human, P) array
        Ortholog expression/features for human, same ortholog order.
    epsilon : float
        Entropic regularization strength for entropic FGW.
    alpha : float
        Weight on the GW structure term. alpha=1 => pure GW, alpha=0 => pure OT on M(w).
    weight_l2 : float
        Strength of quadratic regularization on the weights.
        Larger values push weights toward a more spread-out distribution.
        Smaller values make the solution more selective / sparse.
    loss_fun : str
        POT loss function for the GW term, usually "square_loss".
    metric : str
        Metric used for intra-space distances on PCA embeddings via ot.dist.
    squared : bool
        If True, square the intra-space distances C1 and C2 after ot.dist.
    feature_loss : str
        Per-ortholog cross-species cost. Supported:
          - "sqdiff": (a - b)^2
          - "abs":    |a - b|
    normalize_C : bool
        If True, robustly scale C1 and C2 by median of positive entries.
    normalize_M : bool
        If True, robustly scale M(w) each outer iteration by median of positive entries.
    p, q : arrays or None
        Marginals over mouse / human cells. If None, use uniform.
    init_w : (P,) array or None
        Initial ortholog weights. If None, uses a random Dirichlet initialization.
    max_outer_iter : int
        Maximum number of outer alternating iterations.
    fgw_inner_iter : int
        Number of FGW iterations per outer step.
    fgw_tol : float
        Tolerance passed to POT for the coupling solve.
    outer_tol : float
        Convergence tolerance based on relative changes in T and w.
    solver : str
        POT entropic FGW solver, usually "PGD" or "PPA".
    sinkhorn_warmstart : bool
        Passed to POT; warm-start dual potentials inside Sinkhorn projections.
    random_state : int or None
        Seed for random initialization.
    verbose : bool
        If True, print progress.
    return_log : bool
        If True, also return a log dict.

    Returns
    -------
    T : (n_mouse, n_human) array
        Transport plan.
    w : (P,) array
        Learned nonnegative weights summing to 1.
    log : dict, optional
        Returned only if return_log=True.
    """
    rng = np.random.default_rng(random_state)

    X_mouse_pca = np.asarray(X_mouse_pca, dtype=float)
    X_human_pca = np.asarray(X_human_pca, dtype=float)
    orth_mouse = np.asarray(orth_mouse, dtype=float)
    orth_human = np.asarray(orth_human, dtype=float)

    if X_mouse_pca.ndim != 2 or X_human_pca.ndim != 2:
        raise ValueError("X_mouse_pca and X_human_pca must be 2D arrays.")
    if orth_mouse.ndim != 2 or orth_human.ndim != 2:
        raise ValueError("orth_mouse and orth_human must be 2D arrays.")

    n_mouse = X_mouse_pca.shape[0]
    n_human = X_human_pca.shape[0]

    if orth_mouse.shape[0] != n_mouse:
        raise ValueError(f"orth_mouse must have {n_mouse} rows, got {orth_mouse.shape[0]}.")
    if orth_human.shape[0] != n_human:
        raise ValueError(f"orth_human must have {n_human} rows, got {orth_human.shape[0]}.")
    if orth_mouse.shape[1] != orth_human.shape[1]:
        raise ValueError(
            "orth_mouse and orth_human must have the same number of columns "
            "(same ortholog ordering)."
        )

    P = orth_mouse.shape[1]

    if epsilon <= 0:
        raise ValueError("epsilon must be > 0.")
    if not (0.0 <= alpha <= 1.0):
        raise ValueError("alpha must be in [0, 1].")
    if weight_l2 <= 0:
        raise ValueError("weight_l2 must be > 0.")
    if feature_loss not in {"sqdiff", "abs"}:
        raise ValueError("feature_loss must be one of {'sqdiff', 'abs'}.")

    # ------------------------------------------------------------------
    # Marginals
    # ------------------------------------------------------------------
    if p is None:
        p = np.full(n_mouse, 1.0 / n_mouse, dtype=float)
    else:
        p = np.asarray(p, dtype=float)
        if p.shape != (n_mouse,):
            raise ValueError(f"p must have shape ({n_mouse},), got {p.shape}.")
        if np.any(p < 0):
            raise ValueError("p must be nonnegative.")
        if p.sum() <= 0:
            raise ValueError("p must sum to a positive value.")
        p = p / p.sum()

    if q is None:
        q = np.full(n_human, 1.0 / n_human, dtype=float)
    else:
        q = np.asarray(q, dtype=float)
        if q.shape != (n_human,):
            raise ValueError(f"q must have shape ({n_human},), got {q.shape}.")
        if np.any(q < 0):
            raise ValueError("q must be nonnegative.")
        if q.sum() <= 0:
            raise ValueError("q must sum to a positive value.")
        q = q / q.sum()

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _robust_scale(A: np.ndarray) -> np.ndarray:
        A = np.asarray(A, dtype=float)
        nz = A[A > 0]
        if nz.size == 0:
            return A
        med = np.median(nz)
        if med <= 0:
            return A
        return A / med

    def _project_to_simplex(v: np.ndarray) -> np.ndarray:
        """
        Euclidean projection of v onto the probability simplex:
            {w : w >= 0, sum(w) = 1}
        """
        v = np.asarray(v, dtype=float)
        if v.ndim != 1:
            raise ValueError("Simplex projection expects a 1D array.")

        n = v.size
        u = np.sort(v)[::-1]
        cssv = np.cumsum(u) - 1.0
        ind = np.arange(1, n + 1)
        cond = u - cssv / ind > 0
        if not np.any(cond):
            # Very degenerate case; fallback to uniform
            return np.full(n, 1.0 / n, dtype=float)
        rho = ind[cond][-1]
        theta = cssv[cond][-1] / rho
        w = np.maximum(v - theta, 0.0)
        s = w.sum()
        if s <= 0:
            return np.full(n, 1.0 / n, dtype=float)
        return w / s

    def _solve_weight_qp_on_simplex(c: np.ndarray, lam: float) -> np.ndarray:
        """
        Solve exactly:
            min_w c^T w + (lam/2) ||w||_2^2
            s.t.  w >= 0, sum(w) = 1

        Equivalent to projecting -c/lam onto the simplex.
        """
        return _project_to_simplex(-c / lam)

    def _compute_weighted_M_sqdiff(A: np.ndarray, B: np.ndarray, w: np.ndarray) -> np.ndarray:
        """
        M_ij = sum_k w_k * (A_ik - B_jk)^2
             = (A^2 w)_i + (B^2 w)_j - 2 (A diag(w) B^T)_ij
        """
        Aw = A * w[None, :]
        a2w = (A * A) @ w
        b2w = (B * B) @ w
        M = a2w[:, None] + b2w[None, :] - 2.0 * (Aw @ B.T)
        np.maximum(M, 0.0, out=M)
        return M

    def _compute_weighted_M_abs(A: np.ndarray, B: np.ndarray, w: np.ndarray) -> np.ndarray:
        """
        M_ij = sum_k w_k * |A_ik - B_jk|
        """
        M = np.zeros((A.shape[0], B.shape[0]), dtype=float)
        for k in range(A.shape[1]):
            M += w[k] * np.abs(A[:, [k]] - B[:, k][None, :])
        return M

    def _compute_weighted_M(A: np.ndarray, B: np.ndarray, w: np.ndarray) -> np.ndarray:
        if feature_loss == "sqdiff":
            return _compute_weighted_M_sqdiff(A, B, w)
        return _compute_weighted_M_abs(A, B, w)

    def _weight_objective_coeffs_sqdiff(
        A: np.ndarray,
        B: np.ndarray,
        T: np.ndarray,
    ) -> np.ndarray:
        """
        c_k = sum_ij T_ij * (A_ik - B_jk)^2
        """
        row_mass = T.sum(axis=1)
        col_mass = T.sum(axis=0)
        TB = T @ B

        c = (A * A).T @ row_mass
        c += (B * B).T @ col_mass
        c -= 2.0 * np.sum(A * TB, axis=0)
        return np.maximum(c, 0.0)

    def _weight_objective_coeffs_abs(
        A: np.ndarray,
        B: np.ndarray,
        T: np.ndarray,
    ) -> np.ndarray:
        c = np.empty(A.shape[1], dtype=float)
        for k in range(A.shape[1]):
            Dk = np.abs(A[:, [k]] - B[:, k][None, :])
            c[k] = np.sum(T * Dk)
        return c

    def _weight_objective_coeffs(
        A: np.ndarray,
        B: np.ndarray,
        T: np.ndarray,
    ) -> np.ndarray:
        if feature_loss == "sqdiff":
            return _weight_objective_coeffs_sqdiff(A, B, T)
        return _weight_objective_coeffs_abs(A, B, T)

    # ------------------------------------------------------------------
    # Intra-space costs
    # ------------------------------------------------------------------
    C1 = ot.dist(X_mouse_pca, X_mouse_pca, metric=metric)
    C2 = ot.dist(X_human_pca, X_human_pca, metric=metric)

    if squared:
        C1 = C1 ** 2
        C2 = C2 ** 2

    if normalize_C:
        C1 = _robust_scale(C1)
        C2 = _robust_scale(C2)

    # ------------------------------------------------------------------
    # Initialize weights
    # ------------------------------------------------------------------
    if init_w is None:
        w = rng.dirichlet(np.ones(P, dtype=float))
    else:
        w = np.asarray(init_w, dtype=float)
        if w.shape != (P,):
            raise ValueError(f"init_w must have shape ({P},), got {w.shape}.")
        if np.any(w < 0):
            raise ValueError("init_w must be nonnegative.")
        if w.sum() <= 0:
            raise ValueError("init_w must sum to a positive value.")
        w = w / w.sum()

    # ------------------------------------------------------------------
    # Initialize transport
    # ------------------------------------------------------------------
    T = np.outer(p, q)

    history = {
        "delta_T": [],
        "delta_w": [],
        "feature_obj_linear": [],
        "weight_obj_total": [],
        "weights": [],
    }

    # ------------------------------------------------------------------
    # Outer loop
    # ------------------------------------------------------------------
    for outer_it in range(max_outer_iter):
        w_old = w.copy()
        T_old = T.copy()

        # ---- T step: FGW solve with warm start ----
        M = _compute_weighted_M(orth_mouse, orth_human, w)
        if normalize_M:
            M = _robust_scale(M)

        T = entropic_fused_gromov_wasserstein(
            M=M,
            C1=C1,
            C2=C2,
            p=p,
            q=q,
            loss_fun=loss_fun,
            epsilon=epsilon,
            alpha=alpha,
            G0=T_old,
            max_iter=fgw_inner_iter,
            tol=fgw_tol,
            solver=solver,
            warmstart=sinkhorn_warmstart,
            verbose=verbose,
        )
        T = np.asarray(T, dtype=float)

        if T.shape != (n_mouse, n_human):
            raise RuntimeError(
                f"Unexpected transport shape {T.shape}, expected {(n_mouse, n_human)}."
            )

        # ---- w step: exact quadratic-regularized update on simplex ----
        c = _weight_objective_coeffs(orth_mouse, orth_human, T)
        w = _solve_weight_qp_on_simplex(c, weight_l2)

        # ---- diagnostics ----
        delta_T = np.linalg.norm(T - T_old) / max(np.linalg.norm(T_old), 1e-12)
        delta_w = np.linalg.norm(w - w_old) / max(np.linalg.norm(w_old), 1e-12)

        feature_obj_linear = float(c @ w)
        weight_obj_total = float(c @ w + 0.5 * weight_l2 * np.dot(w, w))

        history["delta_T"].append(float(delta_T))
        history["delta_w"].append(float(delta_w))
        history["feature_obj_linear"].append(feature_obj_linear)
        history["weight_obj_total"].append(weight_obj_total)
        history["weights"].append(w.copy())

        if verbose:
            nnz = int(np.sum(w > 1e-12))
            print(
                f"[outer {outer_it + 1:03d}] "
                f"delta_T={delta_T:.3e}  "
                f"delta_w={delta_w:.3e}  "
                f"nnz(w)={nnz}  "
                f"weight_obj={weight_obj_total:.6e}"
            )

        if max(delta_T, delta_w) < outer_tol:
            if verbose:
                print(f"Converged at outer iteration {outer_it + 1}.")
            break

    if return_log:
        history["final_M"] = M
        history["C1"] = C1
        history["C2"] = C2
        return T, w, history

    return T, w

In [14]:
def extract_ensembl_expression(
    adata,
    ensembl_ids: pd.Series,
    *,
    gene_key: str | None = None,
    fill_missing: float = 0.0,
    return_dense: bool = True,
):
    """
    Extract expression matrix for a given list of Ensembl IDs from an AnnData object.

    Parameters
    ----------
    adata : AnnData
        AnnData object with gene expression matrix (cells x genes).
    ensembl_ids : pd.Series
        Series of Ensembl IDs (order will be preserved in output).
    gene_key : str or None
        If None, match against adata.var_names.
        If provided, match against adata.var[gene_key].
    fill_missing : float
        Value to use for genes not found in adata.
    return_dense : bool
        If True, return a dense numpy array. Otherwise may return sparse.

    Returns
    -------
    X_out : (n_cells, len(ensembl_ids)) array
        Expression matrix aligned to input Ensembl IDs.
    found_mask : (len(ensembl_ids),) bool array
        True where gene was found in adata.
    """
    import scipy.sparse as sp

    if not isinstance(ensembl_ids, pd.Series):
        raise ValueError("ensembl_ids must be a pandas Series.")

    # --- determine gene index source ---
    if gene_key is None:
        gene_index = pd.Index(adata.var_names)
    else:
        if gene_key not in adata.var.columns:
            raise ValueError(f"{gene_key} not found in adata.var.")
        gene_index = pd.Index(adata.var[gene_key])

    # --- map requested genes to indices ---
    lookup = pd.Series(np.arange(len(gene_index)), index=gene_index)
    idx = lookup.reindex(ensembl_ids.values)

    found_mask = idx.notna().values
    valid_idx = idx.dropna().astype(int).values

    n_cells = adata.n_obs
    n_genes = len(ensembl_ids)

    # --- initialize output ---
    if sp.issparse(adata.X):
        X = adata.X
        X_out = np.full((n_cells, n_genes), fill_missing, dtype=float)
        if len(valid_idx) > 0:
            X_sub = X[:, valid_idx]
            if sp.issparse(X_sub):
                X_sub = X_sub.toarray()
            X_out[:, found_mask] = X_sub
    else:
        X = np.asarray(adata.X)
        X_out = np.full((n_cells, n_genes), fill_missing, dtype=float)
        if len(valid_idx) > 0:
            X_out[:, found_mask] = X[:, valid_idx]

    if not return_dense:
        try:
            import scipy.sparse as sp
            X_out = sp.csr_matrix(X_out)
        except ImportError:
            pass

    return X_out, found_mask

In [3]:
human_dir = 'data/tabula_sapiens/'
human_adatas = top_n_organisms_from_species(human_dir, 2, 'human')
human1_adata = human_adatas[1]

In [4]:
mouse_dir = 'data/tabula_muris/'
mouse_adatas = top_n_organisms_from_species(mouse_dir, 2, 'mouse')
mouse1_adata = mouse_adatas[1]

In [5]:
mouse1_matched, human1_matched =  match_cells_by_celltype_tissue(mouse1_adata, human1_adata, cell_type_key='shared_cell_type')

In [6]:
human1_matched.layers["counts"] = human1_matched.X.copy()
sc.pp.normalize_total(human1_matched)
sc.pp.highly_variable_genes(human1_matched, layer="counts", flavor="seurat_v3")            
human1 = human1_matched[:, human1_matched.var['highly_variable']].copy()  
sc.pp.log1p(human1)
human1.layers["norm_logged"] = human1.X.copy()
sc.pp.scale(human1, max_value=10)

In [7]:
mouse1_matched.layers["counts"] = mouse1_matched.X.copy()
sc.pp.normalize_total(mouse1_matched)
sc.pp.highly_variable_genes(mouse1_matched, layer="counts", flavor="seurat_v3")            
mouse1 = mouse1_matched[:, mouse1_matched.var['highly_variable']].copy()  
sc.pp.log1p(mouse1)
mouse1.layers["norm_logged"] = mouse1.X.copy()
sc.pp.scale(mouse1, max_value=10)

In [9]:
correlated_orthologs = ortholog_pearson_r2_by_celltype_biomart(human1, mouse1)

In [10]:
correlated_orthologs.head()

,mouse_gene_name,human_gene_name,slope_through_origin,pearson_r,r2,n_cell_types,mouse_ensembl_id,human_ensembl_id
0,Sfta2,SFTA2,0.997801,1.000000,1.000000,14,ENSMUSG00000090509,ENSG00000196260
1,Slc34a2,SLC34A2,1.136942,0.998023,0.996050,14,ENSMUSG00000029188,ENSG00000157765
2,Bcl11b,BCL11B,1.366328,0.996416,0.992844,14,ENSMUSG00000048251,ENSG00000127152
3,Sftpd,SFTPD,1.135434,0.995731,0.991481,14,ENSMUSG00000021795,ENSG00000133661
4,Sftpb,SFTPB,1.262703,0.995435,0.990892,14,ENSMUSG00000056370,ENSG00000168878


In [11]:
# Why only 154 orthologs?

In [15]:
correlated_orthologs['mouse_ensembl_id']

0      ENSMUSG00000090509
1      ENSMUSG00000029188
2      ENSMUSG00000048251
3      ENSMUSG00000021795
4      ENSMUSG00000056370
              ...        
151    ENSMUSG00000031383
152    ENSMUSG00000024754
153    ENSMUSG00000004698
154    ENSMUSG00000026840
155    ENSMUSG00000037940
Name: mouse_ensembl_id, Length: 156, dtype: object

In [17]:
mouse_orths, mask = extract_ensembl_expression(
    mouse1,
    correlated_orthologs['mouse_ensembl_id'],
    gene_key=None,   # or e.g. "gene_ids" if stored there
)

In [22]:
human_orths, mask = extract_ensembl_expression(
    human1,
    correlated_orthologs['human_ensembl_id'],
    gene_key=None,   # or e.g. "gene_ids" if stored there
)

In [24]:
X_mouse_pca = mouse1.obsm['X_pca']
X_human_pca = human1.obsm['X_pca']

In [36]:
T_opt, w_opt = alternating_weighted_fgw_opt(
    X_mouse_pca,
    X_human_pca,
    mouse_orths,
    human_orths,
    epsilon = 0.01, 
    alpha = 0,
    verbose=True)

It.  |Err         
-------------------
    0|2.102703e-02|
   10|0.000000e+00|
[outer 001] delta_T=1.766e+01  delta_w=4.403e+00  nnz(w)=6  weight_obj=2.847123e-01
It.  |Err         
-------------------
    0|2.106071e-02|
   10|0.000000e+00|
[outer 002] delta_T=1.000e+00  delta_w=9.878e-01  nnz(w)=156  weight_obj=3.205128e-03
It.  |Err         
-------------------
    0|2.125927e-02|
   10|0.000000e+00|
[outer 003] delta_T=2.126e+10  delta_w=6.178e+00  nnz(w)=7  weight_obj=2.069933e-01
It.  |Err         
-------------------
    0|2.125927e-02|
   10|0.000000e+00|
[outer 004] delta_T=1.000e+00  delta_w=9.872e-01  nnz(w)=156  weight_obj=3.205128e-03
It.  |Err         
-------------------
    0|2.125927e-02|
   10|0.000000e+00|
[outer 005] delta_T=2.126e+10  delta_w=6.178e+00  nnz(w)=7  weight_obj=2.069933e-01
It.  |Err         
-------------------
    0|2.125927e-02|
   10|0.000000e+00|
[outer 006] delta_T=1.000e+00  delta_w=9.872e-01  nnz(w)=156  weight_obj=3.205128e-03
It.  |Err       

In [35]:
# Nice! It selected 7 genes.

In [39]:
correlated_orthologs['human_gene_name'][w_opt>0]

0        SFTA2
1      SLC34A2
5       CLDN18
47       GABRD
108     KCNAB3
117      FOXP3
141      SCN3B
Name: human_gene_name, dtype: object

In [41]:
T_opt

array([[8.24534518e-09, 8.99276516e-07, 1.60637923e-10, ...,
        2.50167040e-15, 1.69069957e-14, 2.83187037e-32],
       [1.09072074e-09, 1.03004983e-06, 1.63801508e-10, ...,
        6.99155818e-15, 2.92634618e-16, 1.34661123e-37],
       [3.25965635e-13, 2.54443703e-08, 7.14824118e-10, ...,
        3.26756371e-11, 2.32621016e-11, 1.45616447e-20],
       ...,
       [2.87504800e-18, 1.47447193e-14, 1.25553548e-15, ...,
        1.31763490e-09, 1.70759412e-17, 1.37306797e-26],
       [4.84300356e-14, 2.06851355e-07, 5.25727576e-13, ...,
        3.30895690e-16, 1.47286635e-15, 4.36370946e-34],
       [2.14209530e-17, 9.44863263e-14, 4.80142931e-13, ...,
        7.95450136e-09, 6.85322856e-17, 4.62220342e-23]], shape=(840, 840))

In [42]:
compute_mapping_acc(T_opt, mouse1.obs['shared_cell_type'], human1.obs['shared_cell_type'])

0.45714285714285713

Next step: compare to entropic FGW with just top K orthologs (K=7, K=100)

In [ ]:
# run_entropic_fgw_transport(X_mouse_pca, X_human_pca, ... 